# Logit Attribution

# Logit Attribution

## What This Is
Direct logit attribution (DLA) decomposes the model's final logit score into additive contributions from each component. Since transformers are residual networks:

logit(token) = W_unembed[token] · (sum of all component outputs)

Each component's contribution to the logit is:
contrib_i = W_unembed[token] · component_output_i

This decomposes the prediction into per-component scores, enabling us to identify *which* heads and MLP layers most strongly predict the target token.

In [ ]:
import numpy as np
from typing import Dict, List

np.random.seed(42)

D_MODEL = 16
VOCAB = 40
SEQ = 6
N_LAYERS = 3
N_HEADS = 2
D_HEAD = 8

def softmax(x, axis=-1):
    ex = np.exp(x - x.max(axis=axis, keepdims=True))
    return ex / ex.sum(axis=axis, keepdims=True)

class LogitAttributionModel:
    def __init__(self):
        np.random.seed(7)
        self.W_emb = np.random.randn(VOCAB, D_MODEL) * 0.2
        self.W_pos = np.random.randn(SEQ, D_MODEL) * 0.1
        self.layers = []
        for l in range(N_LAYERS):
            self.layers.append({
                'heads': [{
                    'WQ': np.random.randn(D_MODEL, D_HEAD) * 0.1,
                    'WK': np.random.randn(D_MODEL, D_HEAD) * 0.1,
                    'WV': np.random.randn(D_MODEL, D_HEAD) * 0.1,
                    'WO': np.random.randn(D_HEAD, D_MODEL) * 0.1,
                } for _ in range(N_HEADS)],
                'W_mlp': np.random.randn(D_MODEL, 2*D_MODEL) * 0.1,
                'W_mlp_out': np.random.randn(2*D_MODEL, D_MODEL) * 0.1,
            })
        self.W_unembed = self.W_emb.T * 0.4
        self.component_outputs = []  # cached per-component residual updates
    
    def forward(self, tokens):
        T = len(tokens)
        embed = self.W_emb[tokens] + self.W_pos[:T]
        x = embed.copy()
        self.component_outputs = [('embed', embed.copy())]
        
        for l, layer in enumerate(self.layers):
            for h, head in enumerate(layer['heads']):
                Q = x @ head['WQ']; K = x @ head['WK']; V = x @ head['WV']
                s = Q @ K.T / np.sqrt(D_HEAD)
                s += np.triu(np.ones((T,T))*-1e9, k=1)
                attn = softmax(s)
                head_out = attn @ V @ head['WO']
                x = x + head_out
                self.component_outputs.append((f'L{l}H{h}', head_out.copy()))
            mlp_out = np.maximum(0, x @ layer['W_mlp']) @ layer['W_mlp_out']
            x = x + mlp_out
            self.component_outputs.append((f'L{l}_mlp', mlp_out.copy()))
        
        return x @ self.W_unembed
    
    def direct_logit_attribution(self, tokens, target_token, target_pos):
        logits = self.forward(tokens)
        target_dir = self.W_unembed[:, target_token]
        
        attributions = []
        for name, output in self.component_outputs:
            contrib = float(output[target_pos] @ target_dir)
            attributions.append((name, contrib))
        return attributions, float(logits[target_pos, target_token])

model = LogitAttributionModel()
tokens = [5, 12, 3, 8, 20, 7]
target_token = 15
target_pos = 5

attribs, final_logit = model.direct_logit_attribution(tokens, target_token, target_pos)

print(f'Direct Logit Attribution for token {target_token} at position {target_pos}')
print(f'Final logit: {final_logit:.4f}')
print(f'Sum of attributions: {sum(a for _, a in attribs):.4f}')
print('\nComponent contributions (sorted by |contribution|):')
for name, contrib in sorted(attribs, key=lambda x: abs(x[1]), reverse=True)[:8]:
    pct = contrib / (abs(final_logit) + 1e-8) * 100
    print(f'  {name:<12}: {contrib:+.4f} ({pct:+.1f}%)')
